# QECCT 재현 실험 (Deep Quantum Error Correction, AAAI-24)
**논문**: Choukroun & Wolf, "Deep Quantum Error Correction", AAAI-24  
**목표**: QECCT(Quantum Error Correction Code Transformer) 모델 및 실험 환경을 최대한 동일하게 재현

## 주요 구현 사항
1. Toric Code 생성 (패리티 체크 행렬, 논리 연산자, 어텐션 마스크)
2. 노이즈 모델 (Independent, Depolarization)
3. 초기 노이즈 추정기 g_ω
4. QECCT Transformer 아키텍처 (Masked Self-Attention, Average Pooling)
5. 손실 함수 (L_BER + L_LER + L_g, 미분 가능한 LER)
6. MWPM 베이스라인 비교


In [ ]:
# 필요한 패키지 설치 (pymatching이 없는 경우)
# !pip install pymatching

import sys
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 모듈 임포트
from qecct_models import ToricCode, QECCT, QECCTLoss, NoiseEstimator, compute_ber, compute_ler
from qecct_train import Trainer, evaluate_mwpm, plot_ler_comparison, plot_training_history, plot_multi_L_comparison

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 1. 실험 설정 (Configuration)

논문의 기본 하이퍼파라미터를 따르되, 계산 자원에 따라 조정 가능하도록 설계.

| 파라미터 | 논문 기본값 | 조정 가능 |
|----------|------------|----------|
| N (Transformer layers) | 6 | ✓ |
| d (embedding dim) | 128 | ✓ |
| n_heads | 8 | ✓ |
| batch_size | 512 | ✓ |
| lr_init | 5e-4 | ✓ |
| lr_min | 5e-7 | ✓ |
| λ_BER | 0.5 | ✓ |
| λ_LER | 1.0 | ✓ |
| λ_g | 0.5 | ✓ |
| epochs | 200-800 | ✓ |
| batches/epoch | 5000 | ✓ |


In [ ]:
# ============================================================
# 실험 설정 - 이 셀의 값을 수정하여 실험 조건을 변경
# ============================================================

CONFIG = {
    # --- 모델 아키텍처 ---
    'N_layers': 6,          # Transformer 레이어 수
    'd_model': 128,         # 임베딩 차원
    'n_heads': 8,           # 어텐션 헤드 수

    # --- 학습 설정 ---
    'lr': 5e-4,             # 초기 학습률
    'lr_min': 5e-7,         # 최소 학습률 (cosine decay)
    'batch_size': 512,      # 미니배치 크기
    'n_epochs': 200,        # 에폭 수 (논문: 200-800)
    'n_batches': 5000,      # 에폭당 미니배치 수

    # --- 손실 가중치 ---
    'lambda_ber': 0.5,
    'lambda_ler': 1.0,
    'lambda_g': 0.5,

    # --- 실험 조건 ---
    'lattice_sizes': [3, 4, 5],     # 테스트할 격자 크기 L (논문: 2~10)
    'noise_type': 'independent',     # 'independent' 또는 'depolarization'
    'p_train_range': (0.01, 0.15),   # 학습 시 노이즈 범위
    'p_eval': [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14],  # 평가 에러율

    # --- 평가 ---
    'eval_every': 20,        # 몇 에폭마다 평가
    'eval_n_samples': 10000, # 평가 샘플 수 (논문: 10^6)
    'mwpm_n_samples': 10000, # MWPM 베이스라인 샘플 수

    # --- Quick-run 모드 (빠른 검증용) ---
    'quick_run': True,       # True면 축소된 설정으로 실행
}

# Quick-run 모드: 빠른 코드 검증을 위한 축소 설정
if CONFIG['quick_run']:
    CONFIG.update({
        'N_layers': 3,
        'd_model': 64,
        'n_heads': 4,
        'n_epochs': 5,
        'n_batches': 50,
        'lattice_sizes': [3],
        'eval_every': 5,
        'eval_n_samples': 500,
        'mwpm_n_samples': 500,
    })
    print("⚡ Quick-run mode enabled (축소된 설정)")

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 2. Toric Code 검증

Toric Code의 패리티 체크 행렬 H, 논리 연산자 L, 어텐션 마스크를 생성하고 검증합니다.


In [ ]:
# Toric Code 생성 및 기본 검증
L_test = 3
code = ToricCode(L_test)

print(f"=== Toric Code L={L_test} ===")
print(f"Physical qubits (n): {code.n}")
print(f"Syndrome bits (n_s): {code.n_s}")
print(f"Logical qubits (k): {code.k}")
print(f"H shape: {code.H.shape}")
print(f"L shape: {code.L_matrix.shape}")
print(f"Mask shape: {code.mask.shape}")

# 검증 1: H의 각 행은 정확히 4개의 1을 가져야 함 (4-body stabilizer)
row_sums = code.H.sum(axis=1)
print(f"\nH row sums (should be 4): min={row_sums.min()}, max={row_sums.max()}")

# 검증 2: 알려진 노이즈에 대한 신드롬 계산
noise = np.zeros(code.n, dtype=np.float32)
noise[0] = 1  # Single X error on first horizontal edge
syndrome = code.get_syndrome(noise)
print(f"\nSingle-error syndrome: {np.sum(syndrome)} nonzero elements (expected: 2)")

# 검증 3: 마스크 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(code.H, cmap='binary', aspect='auto')
axes[0].set_title(f'Parity Check Matrix H ({code.H.shape[0]}×{code.H.shape[1]})')
axes[1].imshow(code.mask, cmap='binary', aspect='auto')
axes[1].set_title(f'Attention Mask ({code.mask.shape[0]}×{code.mask.shape[1]})')
axes[2].imshow(code.L_matrix, cmap='binary', aspect='auto')
axes[2].set_title(f'Logical Operators L ({code.L_matrix.shape[0]}×{code.L_matrix.shape[1]})')
plt.tight_layout()
plt.show()

print("\n✅ Toric Code 검증 완료")


## 3. 노이즈 모델 검증

In [ ]:
# 노이즈 모델 검증
L_test = 3
code = ToricCode(L_test)
p_test = 0.1

# Independent noise
ind_noise = code.sample_independent_noise(p_test, 10000)
measured_rate = ind_noise.mean()
print(f"Independent noise (p={p_test}): measured rate = {measured_rate:.4f}")

# Depolarization noise
dep_noise = code.sample_depolarization_noise(p_test, 10000)
measured_rate_dep = dep_noise.mean()
print(f"Depolarization noise (p={p_test}): measured rate = {measured_rate_dep:.4f}")

# Syndrome distribution
syndromes = np.array([code.get_syndrome(ind_noise[i]) for i in range(10000)])
print(f"Average syndrome weight: {syndromes.sum(axis=1).mean():.2f}")
print("\n✅ 노이즈 모델 검증 완료")


## 4. QECCT 모델 아키텍처 검증

In [ ]:
# QECCT 모델 생성 및 검증 (작은 크기)
L_test = 3
code = ToricCode(L_test)

model = QECCT(
    code=code,
    N=CONFIG['N_layers'],
    d_model=CONFIG['d_model'],
    n_heads=CONFIG['n_heads'],
    use_faulty=False
).to(device)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"=== QECCT Model (L={L_test}) ===")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass 테스트
noise = code.sample_independent_noise(0.1, 4)
syndromes = np.array([code.get_syndrome(noise[i]) for i in range(4)])
s_tensor = torch.tensor(syndromes, dtype=torch.float32).to(device)

output = model(s_tensor)
print(f"\nInput syndrome shape: {s_tensor.shape}")
print(f"Output prediction shape: {output['prediction'].shape}")
print(f"Noise estimate shape: {output['noise_est'].shape}")
print(f"Prediction range: [{output['prediction'].min():.4f}, {output['prediction'].max():.4f}]")

# 손실 함수 테스트
criterion = QECCTLoss(
    L_matrix=torch.tensor(code.L_matrix, dtype=torch.float32).to(device),
    lambda_ber=CONFIG['lambda_ber'],
    lambda_ler=CONFIG['lambda_ler'],
    lambda_g=CONFIG['lambda_g']
).to(device)

noise_tensor = torch.tensor(noise, dtype=torch.float32).to(device)
losses = criterion(output['prediction'], output['noise_est'], noise_tensor)
print(f"\nLoss values:")
print(f"  Total: {losses['total'].item():.4f}")
print(f"  BER:   {losses['ber'].item():.4f}")
print(f"  LER:   {losses['ler'].item():.4f}")
print(f"  G:     {losses['g'].item():.4f}")

# Gradient flow 확인
losses['total'].backward()
grad_norms = []
for name, p in model.named_parameters():
    if p.grad is not None:
        grad_norms.append((name, p.grad.norm().item()))
print(f"\nGradient flow check: {len(grad_norms)}/{trainable_params} params have gradients")
print(f"Max gradient norm: {max(g for _, g in grad_norms):.6f}")
print(f"Min gradient norm: {min(g for _, g in grad_norms):.6f}")
print("\n✅ QECCT 모델 검증 완료")


## 5. 학습 실행

각 격자 크기 L에 대해 QECCT 모델을 학습합니다.


In [ ]:
# 학습 실행
all_results = {}
trained_models = {}

for L in CONFIG['lattice_sizes']:
    print(f"\n{'='*60}")
    print(f" Training QECCT for Toric Code L={L} (n={2*L*L})")
    print(f"{'='*60}")

    code = ToricCode(L)

    model = QECCT(
        code=code,
        N=CONFIG['N_layers'],
        d_model=CONFIG['d_model'],
        n_heads=CONFIG['n_heads'],
        use_faulty=False
    ).to(device)

    trainer = Trainer(
        model=model,
        code=code,
        device=device,
        lr=CONFIG['lr'],
        lr_min=CONFIG['lr_min'],
        batch_size=CONFIG['batch_size'],
        noise_type=CONFIG['noise_type'],
        p_range=CONFIG['p_train_range'],
        lambda_ber=CONFIG['lambda_ber'],
        lambda_ler=CONFIG['lambda_ler'],
        lambda_g=CONFIG['lambda_g']
    )

    history = trainer.train(
        n_epochs=CONFIG['n_epochs'],
        n_batches_per_epoch=CONFIG['n_batches'],
        eval_every=CONFIG['eval_every'],
        eval_p_range=CONFIG['p_eval'],
        eval_n_samples=CONFIG['eval_n_samples'],
        verbose=True
    )

    # 최종 평가
    final_results = trainer.evaluate(CONFIG['p_eval'], CONFIG['eval_n_samples'])

    all_results[L] = {
        'qecct': final_results,
        'history': history
    }
    trained_models[L] = model

    # 학습 곡선 출력
    plot_training_history(history)

    print(f"\n✅ L={L} 학습 완료")


## 6. MWPM 베이스라인 평가

In [ ]:
# MWPM 평가 (pymatching 필요)
for L in CONFIG['lattice_sizes']:
    print(f"\nEvaluating MWPM for L={L}...")
    code = ToricCode(L)
    mwpm_results = evaluate_mwpm(
        code, CONFIG['p_eval'],
        noise_type=CONFIG['noise_type'],
        n_samples=CONFIG['mwpm_n_samples']
    )
    all_results[L]['mwpm'] = mwpm_results

    # LER 비교 출력
    print(f"\n  {'p':>6s}  {'MWPM LER':>10s}  {'QECCT LER':>10s}")
    print(f"  {'-'*30}")
    for i, p in enumerate(CONFIG['p_eval']):
        m_ler = mwpm_results['ler'][i]
        q_ler = all_results[L]['qecct']['ler'][i]
        print(f"  {p:6.3f}  {m_ler:10.6f}  {q_ler:10.6f}")

print("\n✅ MWPM 베이스라인 평가 완료")


## 7. 결과 비교 시각화

In [ ]:
# 개별 L별 LER 비교
for L in CONFIG['lattice_sizes']:
    if 'mwpm' in all_results[L]:
        plot_ler_comparison(
            all_results[L]['qecct'],
            all_results[L]['mwpm'],
            L, CONFIG['noise_type']
        )

# 전체 비교
plot_multi_L_comparison(all_results, metric='ler')
plot_multi_L_comparison(all_results, metric='ber')


## 8. 결과 저장

In [ ]:
# 결과 저장
import json
from datetime import datetime

save_data = {
    'config': {k: str(v) if not isinstance(v, (int, float, bool, str, list)) else v 
                for k, v in CONFIG.items()},
    'timestamp': datetime.now().isoformat(),
    'device': str(device),
}

for L in CONFIG['lattice_sizes']:
    save_data[f'L{L}'] = {
        'qecct_ler': all_results[L]['qecct']['ler'],
        'qecct_ber': all_results[L]['qecct']['ber'],
    }
    if 'mwpm' in all_results[L]:
        save_data[f'L{L}']['mwpm_ler'] = all_results[L]['mwpm']['ler']
        save_data[f'L{L}']['mwpm_ber'] = all_results[L]['mwpm']['ber']
    save_data[f'L{L}']['p_values'] = CONFIG['p_eval']

with open('results.json', 'w') as f:
    json.dump(save_data, f, indent=2)
print("Results saved to results.json")

# 모델 저장
for L, model in trained_models.items():
    torch.save(model.state_dict(), f'qecct_L{L}.pt')
    print(f"Model saved: qecct_L{L}.pt")

print("\n✅ 모든 결과 저장 완료")


## 9. 실험 요약

### 재현된 논문 핵심 요소
- **QECCT 아키텍처**: Masked Self-Attention Transformer + 초기 노이즈 추정기
- **손실 함수**: L_BER + L_LER (bipolar mapping으로 미분 가능) + L_g
- **Toric Code**: 2D 격자 기반 위상 양자 오류 정정 코드
- **노이즈 모델**: Independent / Depolarization
- **베이스라인**: MWPM (Minimum Weight Perfect Matching)

### 논문과의 차이점
- GPU 자원 제약으로 코드 길이 L과 에폭 수를 축소하여 실험
- 평가 샘플 수: 10^4 (논문: 10^6)
- Faulty syndrome 실험은 별도로 진행 가능 (`use_faulty=True` 설정)

### 다음 단계
1. **온디바이스 SoC 도메인 적응**: 모델 경량화 (Pruning, Quantization, Distillation)
2. **Linear Attention 적용**: Self-Attention O(n²) → O(n) 복잡도 감소
3. **하드웨어 최적화**: Layer Fusion, Weight Sharing
